# 03 — Financial Trend Detection

Analyze monthly trends, detect risky patterns, and compute a financial health score.

In [0]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.data_loader import load_data, preprocess, get_user_data
from src.trend_detection import (
    monthly_trends, spending_trend, detect_risky_patterns,
    financial_health_score, category_trend
)

In [0]:
df = preprocess(load_data('../data/virtual_financial_advisor_data.csv'))
user_df = get_user_data(df, 'user_1')
print(f'user_1 transactions: {len(user_df)}')

## 1. Monthly Trends

In [0]:
trends = monthly_trends(user_df)
print(f'Months of data: {len(trends)}')
trends.head()

In [0]:
fig = px.line(trends, x='month', y=['income', 'expenses', 'net_savings'],
              title='user_1 — Monthly Income, Expenses & Net Savings',
              labels={'value': 'Amount ($)', 'month': 'Month'})
fig.show()

## 2. Spending Trend (Rolling Average)

In [0]:
sp = spending_trend(user_df, window=3)
fig2 = px.line(sp, x='month', y=['expenses', 'rolling_avg'],
               title='user_1 — Spending Trend (3-month rolling avg)')
fig2.show()
sp[['month', 'expenses', 'rolling_avg', 'trend_direction']].tail(6)

## 3. Risk Alerts

In [0]:
risks = detect_risky_patterns(user_df)
if risks:
    for r in risks:
        print(f"[{r['severity'].upper()}] {r['risk']}: {r['detail']}")
else:
    print('No significant risks detected.')

## 4. Financial Health Score

In [0]:
health = financial_health_score(user_df)
print(f"Overall Score: {health['score']}/100")
print('\nBreakdown:')
for k, v in health['breakdown'].items():
    print(f'  {k}: {v}')

In [0]:
# Gauge chart
fig3 = go.Figure(go.Indicator(
    mode='gauge+number',
    value=health['score'],
    gauge={'axis': {'range': [0, 100]},
           'steps': [
               {'range': [0, 40], 'color': '#ff4b4b'},
               {'range': [40, 70], 'color': '#ffa500'},
               {'range': [70, 100], 'color': '#00cc96'}]},
    title={'text': 'Financial Health Score'}))
fig3.show()

## 5. Category-Level Trend

In [0]:
for cat in ['Groceries', 'Dining', 'Entertainment']:
    ct = category_trend(user_df, cat)
    fig_cat = px.bar(ct, x='month', y='total', title=f'user_1 — {cat} Monthly Spend')
    fig_cat.show()